In [45]:
import numpy as np
import pandas as pd
from datetime import date
from src.download_data import get_data

In [46]:
data = get_data()
data.head(5)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher
0,3057270,Seafarer's Gambit,2024,"Jul 5, 2024",Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios
1,3822840,Capitalist Misadventures,2025,"Jul 25, 2025",Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios
2,3216640,The Beast and the Princess,2025,"Jun 17, 2025",Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames
3,2403620,Air Twister,2023,"Nov 10, 2023",Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ
4,1538040,Horde Slayer,2021,"Mar 19, 2021",Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues


In [47]:
# Validate nulls
print(data.info())
null_before_clean = data.isnull().sum()
display('-Validation nulls count:-', null_before_clean)

# Handle nulls
data.dropna(subset=['genres', 'categories'], inplace=True)
data.fillna('Unknown', inplace=True)
data.reset_index(drop=True, inplace=True)

null_after_clean = data.isnull().sum()
display('-Handled nulls count:-', null_after_clean)

<class 'pandas.DataFrame'>
RangeIndex: 65521 entries, 0 to 65520
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appid            65521 non-null  int64  
 1   name             65521 non-null  str    
 2   release_year     65521 non-null  int64  
 3   release_date     65521 non-null  str    
 4   genres           65455 non-null  str    
 5   categories       65514 non-null  str    
 6   price            65521 non-null  float64
 7   recommendations  65521 non-null  int64  
 8   developer        65468 non-null  str    
 9   publisher        65338 non-null  str    
dtypes: float64(1), int64(3), str(6)
memory usage: 5.0 MB
None


'-Validation nulls count:-'

appid                0
name                 0
release_year         0
release_date         0
genres              66
categories           7
price                0
recommendations      0
developer           53
publisher          183
dtype: int64

'-Handled nulls count:-'

appid              0
name               0
release_year       0
release_date       0
genres             0
categories         0
price              0
recommendations    0
developer          0
publisher          0
dtype: int64

In [48]:
# Validate 'name' column
data['name'] = data['name'].str.strip()

In [49]:
# Validate 'release_year' and  'release_date' columns
data_years = data['release_year'].unique()
study_years = [2021, 2022, 2023, 2024, 2025]
if sorted(data_years) != study_years:
    data = data[data['release_year'].isin(study_years)]
print(f"Unique years in 'release_year': {data_years}")

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
print(f"Number of missing release dates: {data['release_date'].isna().sum()}")
data.dropna(subset=['release_date'], inplace=True)

Unique years in 'release_year': [2024 2025 2023 2021 2022]
Number of missing release dates: 1399


In [50]:
# Validate 'genres' and 'categories' columns
genres_dummies = data['genres'].str.get_dummies(sep=';').astype('Sparse[uint8]')
categories_dummies = data['categories'].str.get_dummies(sep=';').astype('Sparse[uint8]')
data = pd.concat([data, genres_dummies, categories_dummies], axis=1)
data.head(5)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher,...,Steam Turn Notifications,Steam Workshop,Stereo Sound,Subtitle Options,Surround Sound,Touch Only Option,Tracked Controller Support,VR Only,VR Support,VR Supported
0,3057270,Seafarer's Gambit,2024,2024-07-05,Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios,...,0,0,0,0,0,0,0,0,0,0
1,3822840,Capitalist Misadventures,2025,2025-07-25,Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios,...,0,0,0,0,0,0,0,0,0,0
2,3216640,The Beast and the Princess,2025,2025-06-17,Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames,...,0,0,0,0,0,0,0,0,0,0
3,2403620,Air Twister,2023,2023-11-10,Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ,...,0,0,0,0,0,0,0,0,0,0
4,1538040,Horde Slayer,2021,2021-03-19,Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues,...,0,0,0,0,0,0,0,0,0,0


In [51]:
# Validate 'price' and 'recommendations' columns
negative_prices = data['price'] < 0
negative_recommendations = data['recommendations'] < 0
data = data[~(negative_prices | negative_recommendations)]
print(f"Removed {negative_prices.sum()} negative prices and {negative_recommendations.sum()} negative recommendations")

Removed 0 negative prices and 0 negative recommendations


In [52]:
# Validate 'developer' and 'publisher' columns
data['developer'] = data['developer'].str.strip()
data['publisher'] = data['publisher'].str.strip()